# Lab 09-09 · The Long-Channel MOSFET

Companion to chapter [09-09](../../chapters/09-09-long-channel-mosfet.md). Handout: [README.md](README.md).

The transistor: $W$ = 10 µm, $L$ = 1 µm, $T_{ox}$ = 4 nm, $N_A$ = 5 × 10¹⁷ cm⁻³, $V_t$ = 0.4 V, at 300 K.

In [1]:
import math
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q = 1.602176634e-19
VT = 1.380649e-23 * 300 / Q
NI = 1e10
EPS = 11.7 * 8.8541878128e-14
EOX = 3.9 * 8.8541878128e-14
W, L, TOX, NA, VT0 = 10e-4, 1e-4, 4e-7, 5e17, 0.4
COX = EOX / TOX
PB = VT * math.log(NA / NI)
WD = math.sqrt(2 * EPS * 2 * PB / (Q * NA))
M = 1 + EPS / WD / COX
GAM = math.sqrt(2 * Q * NA * EPS) / COX
mu_univ = lambda vgs: 540 / (1 + ((vgs + VT0 + 0.2) / (6 * TOX) / 0.9e6) ** 1.85)
MU = mu_univ(1.2)
print(f"m = {M:.4f}, gamma = {GAM:.4f} V^0.5, mu = {MU:.1f} cm^2/Vs")
assert abs(M - 1.246) < 5e-4 and abs(MU - 315.1) < 0.05

m = 1.2465, gamma = 0.4719 V^0.5, mu = 315.1 cm^2/Vs


## Part 1 — The channel

Current continuity: $I_D\,dx = W\mu C_{ox}q(V_c)\,dV_c$, where $q$ is the local inversion charge divided by $C_{ox}$. Integrating over $V_c$ gives both the current and the position $x(V_c)$.

*Predict first:* at $V_{DS} = V_{Dsat}$, what fraction of the channel has less than half its source charge?

In [2]:
def current(vds, q):
    return W / L * MU * COX * quad(q, 0, vds, epsabs=0, epsrel=1e-12, limit=200)[0]
vov = 1.2 - VT0
q_m = lambda V: vov - M * V
VFB = VT0 - 2 * PB - GAM * math.sqrt(2 * PB)
q_b = lambda V: 1.2 - VFB - 2 * PB - V - GAM * math.sqrt(2 * PB + V)
vsat_m = vov / M
vsat_b = brentq(q_b, 0, 2)
I_m, I_b = current(vsat_m, q_m), current(vsat_b, q_b)
print(f"m-model: V_Dsat = {vsat_m:.4f} V, I_Dsat = {I_m*1e3:.4f} mA;  exact: V_Dsat = {vsat_b:.4f} V, I_Dsat = {I_b*1e3:.4f} mA")
assert abs(I_m * 1e3 - 0.698) < 5e-4 and abs(I_b * 1e3 - 0.7118) < 5e-5
# position where the charge has halved, at saturation
Vh = vov / (2 * M)
xh = W * MU * COX * quad(q_m, 0, Vh)[0] / I_m
print(f"charge falls below half its source value beyond x/L = {xh/L:.3f}")
V = np.linspace(0, vsat_m * 0.9999, 300)
x = np.array([W * MU * COX * quad(q_m, 0, v)[0] / I_m for v in V])
plt.plot(x / L, [q_m(v) / vov for v in V]); plt.xlabel("x/L"); plt.ylabel("Q_inv / C_ox V_ov"); plt.show()

m-model: V_Dsat = 0.6418 V, I_Dsat = 0.6984 mA;  exact: V_Dsat = 0.6594 V, I_Dsat = 0.7118 mA
charge falls below half its source value beyond x/L = 0.750


## Part 2 — Universal mobility

*Predict first:* with mobility falling at high gate voltage, does $\sqrt{I_{Dsat}}$ against $V_{GS}$ bend up or down?

In [3]:
vg = np.linspace(0.5, 2.0, 61)
i_const = np.array([W / L * MU * COX * (v - VT0) ** 2 / (2 * M) for v in vg])
i_univ = np.array([W / L * mu_univ(v) * COX * (v - VT0) ** 2 / (2 * M) for v in vg])
plt.plot(vg, np.sqrt(i_const), label="constant mobility"); plt.plot(vg, np.sqrt(i_univ), label="universal mobility")
plt.xlabel("V_GS (V)"); plt.ylabel("sqrt(I_Dsat)"); plt.legend(); plt.show()
print(f"at V_GS = 2.0 V the universal-mobility current is {i_univ[-1]/i_const[-1]:.3f} of the constant-mobility value")

at V_GS = 2.0 V the universal-mobility current is 0.711 of the constant-mobility value


## Part 3 — $g_m/I_D$ from weak to strong inversion

Use the EKV-form interpolation $I_D = 2MKV_T^2\ln^2(1 + e^{V_{ov}/2MV_T})$.

*Predict first (the chapter's guided exercise):* at what overdrive does $g_m/I_D$ fall to half of $1/(MV_T)$?

In [4]:
K = W / L * MU * COX
I_ekv = lambda vov: 2 * M * K * VT**2 * math.log(1 + math.exp(vov / (2 * M * VT)))**2
def gm_id(vov, h=1e-5):
    return (math.log(I_ekv(vov + h)) - math.log(I_ekv(vov - h))) / (2 * h)
vh = brentq(lambda v: gm_id(v) - 0.5 / (M * VT), -0.2, 1.0)
print(f"weak-inversion limit 1/(m V_T) = {1/(M*VT):.2f} 1/V; halves at V_ov = {vh*1e3:.1f} mV; at V_ov = 0: {gm_id(0):.2f} 1/V")
assert abs(vh * 1e3 - 88.1) < 0.05 and abs(gm_id(0) - 22.39) < 0.005
vv = np.linspace(-0.3, 1.0, 200)
plt.plot(vv, [gm_id(v) for v in vv], label="interpolation"); plt.plot(vv[vv > 0.05], [2 / v for v in vv[vv > 0.05]], "--", label="2/V_ov")
plt.axhline(1 / (M * VT), ls=":", color="k"); plt.ylim(0, 40); plt.xlabel("V_ov (V)"); plt.ylabel("g_m/I_D (1/V)"); plt.legend(); plt.show()

weak-inversion limit 1/(m V_T) = 31.03 1/V; halves at V_ov = 88.1 mV; at V_ov = 0: 22.39 1/V


## Questions

Answer the discussion questions in the [lab handout](README.md), then try the optional extension (series resistance).